# ❤️ Heart Attack Risk Analysis & Prediction
**Author:** Parnil Kashyap  
**Dataset:** Cleveland Heart Disease — UCI Machine Learning Repository (294 patients, 13 features)  

---

This notebook mirrors the master application `Parnil_Kashyap_HeartAttackRiskAnalysis.py`  
and covers the complete data science pipeline:

| # | Section |
|---|--------|
| 1 | Setup & Data Loading |
| 2 | Data Preprocessing |
| 3 | Exploratory Data Analysis (EDA) |
| 4 | Feature Engineering & Scaling |
| 5 | Model Training — Random Forest & Logistic Regression |
| 6 | Model Evaluation — Accuracy, AUC, Confusion Matrix, ROC |
| 7 | Feature Importance |
| 8 | Live Prediction Demo |
| 9 | Key Findings & Recommendations |

## 1. Setup & Data Loading

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
import joblib

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

DATA_PATH = os.path.join('data', 'data.csv')
df_raw = pd.read_csv(DATA_PATH, na_values=['?'])
df_raw.columns = df_raw.columns.str.strip()
print(f'Loaded: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns')
df_raw.head(10)

In [ ]:
print('=== Column dtypes ===')
print(df_raw.dtypes)
print('\n=== Missing values per column ===')
mv = df_raw.isnull().sum()
print(mv[mv > 0])
print('\n=== Descriptive statistics ===')
df_raw.describe().round(2)

## 2. Data Preprocessing

In [ ]:
df = df_raw.copy()

# Binarise target: 0 = no disease, 1 = disease
df['target'] = (df['num'] > 0).astype(int)
df.drop(columns=['num'], inplace=True)

# Coerce all feature columns to numeric (handles any stray strings)
for col in df.columns:
    if col != 'target':
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Impute NaN with column median (CoW-safe, pandas 2.x compatible)
for col in df.select_dtypes(include=[np.number]).columns:
    df[col] = df[col].fillna(df[col].median())

df = df.dropna().reset_index(drop=True)

print(f'Clean shape: {df.shape}')
print(f'\nTarget distribution:')
print(df['target'].value_counts().rename({0: 'No Disease (0)', 1: 'Disease (1)'}))
print(f'\nPrevalence: {df["target"].mean()*100:.1f}%')

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# 3.1 — Target class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['target'].value_counts().sort_index()
axes[0].pie(counts, labels=['No Disease', 'Disease'],
            colors=['#2ecc71', '#e74c3c'], autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title(f'Target Class Split  (n={len(df)})')

sns.countplot(x='target', data=df, palette=['#2ecc71', '#e74c3c'], ax=axes[1])
axes[1].set_xticklabels(['No Disease (0)', 'Disease (1)'])
axes[1].set_xlabel(''); axes[1].set_title('Patient Count by Class')

plt.suptitle('Target Variable Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 3.2 — Age distributions & age-group risk rate
CP_LABELS = {0:'Typical Angina',1:'Atypical Angina',2:'Non-Anginal',3:'Asymptomatic'}

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for t, c, lbl in [(0,'#2ecc71','No Disease'),(1,'#e74c3c','Disease')]:
    axes[0].hist(df[df.target==t]['age'], bins=15,
                 alpha=0.72, color=c, label=lbl, edgecolor='white')
axes[0].set_xlabel('Age (years)'); axes[0].set_ylabel('Count')
axes[0].legend(); axes[0].set_title('Age Distribution by Risk Class')

df['age_group'] = pd.cut(df['age'], bins=[20,35,45,55,65,80],
                          labels=['20-35','35-45','45-55','55-65','65-80'])
ag = df.groupby('age_group', observed=True)['target'].mean()*100
bar_c = ['#e74c3c' if v > 40 else '#2ecc71' for v in ag]
ag.plot(kind='bar', color=bar_c, ax=axes[1], edgecolor='black', width=0.55)
axes[1].set_xlabel('Age Group'); axes[1].set_ylabel('Disease Rate (%)')
axes[1].set_title('Heart Disease Rate by Age Group')
axes[1].tick_params(axis='x', rotation=0); axes[1].set_ylim(0,100)

plt.tight_layout(); plt.show()
df.drop(columns=['age_group'], inplace=True)

In [ ]:
# 3.3 — Sex and Chest Pain risk rates
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sex_risk = df.groupby('sex')['target'].mean()*100
sex_risk.index = ['Female','Male']
sex_risk.plot(kind='bar', color=['#f39c12','#3498db'],
              ax=axes[0], edgecolor='black', width=0.45)
axes[0].set_ylabel('Disease Rate (%)'); axes[0].set_title('Disease Rate by Sex')
axes[0].tick_params(axis='x', rotation=0); axes[0].set_ylim(0,100)

cp_risk = df.groupby('cp')['target'].mean()*100
cp_risk.index = [CP_LABELS[i] for i in cp_risk.index]
cp_risk.plot(kind='bar', color=['#9b59b6','#e74c3c','#3498db','#e67e22'],
             ax=axes[1], edgecolor='black', width=0.55)
axes[1].set_ylabel('Disease Rate (%)'); axes[1].set_title('Disease Rate by Chest Pain Type')
axes[1].tick_params(axis='x', rotation=20); axes[1].set_ylim(0,100)

plt.tight_layout(); plt.show()
print(f'\nSex risk rates:\n{sex_risk.round(1)}')
print(f'\nChest pain risk rates:\n{cp_risk.round(1)}')

In [ ]:
# 3.4 — Clinical feature boxplots by target
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, feat, lbl in zip(
    axes,
    ['chol','thalach','oldpeak'],
    ['Cholesterol (mg/dl)','Max Heart Rate (thalach)','ST Depression (oldpeak)']
):
    df.boxplot(column=feat, by='target', ax=ax,
               boxprops=dict(color='#3498db'),
               medianprops=dict(color='#e74c3c', linewidth=2.5),
               whiskerprops=dict(color='#3498db'),
               capprops=dict(color='#3498db'))
    ax.set_xlabel('Target  (0=No Disease, 1=Disease)')
    ax.set_title(lbl)
plt.suptitle('Clinical Feature Distributions by Heart Disease Status',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# 3.5 — Correlation heatmap (lower triangle)
fig, ax = plt.subplots(figsize=(11, 8))
corr = df.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', linewidths=0.45, ax=ax,
            annot_kws={'size': 8})
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

print('\nTarget correlations (sorted):')
print(corr['target'].drop('target').sort_values(ascending=False).round(3))

In [ ]:
# 3.6 — Max Heart Rate vs Age scatter
fig, ax = plt.subplots(figsize=(9, 5))
c_map = {0:'#2ecc71', 1:'#e74c3c'}
for t_val, grp in df.groupby('target'):
    ax.scatter(grp['age'], grp['thalach'],
               c=c_map[t_val], alpha=0.65,
               edgecolors='k', linewidths=0.3, s=50,
               label='No Disease' if t_val == 0 else 'Disease')
ax.set_xlabel('Age (years)'); ax.set_ylabel('Max Heart Rate (thalach)')
ax.set_title('Max Heart Rate vs Age', fontsize=12)
ax.legend()
plt.tight_layout(); plt.show()

## 4. Feature Engineering & Scaling

In [ ]:
FEATURES = ['age','sex','cp','trestbps','chol','fbs',
            'restecg','thalach','exang','oldpeak','slope','ca','thal']
FEATURES = [f for f in FEATURES if f in df.columns]

X = df[FEATURES]
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]} samples  |  Test: {X_test.shape[0]} samples')
print(f'Features ({len(FEATURES)}): {FEATURES}')

## 5. Model Training

In [ ]:
# ── Random Forest ─────────────────────────────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=200, max_depth=8,
    random_state=42, class_weight='balanced')
rf.fit(X_train_sc, y_train)

# ── Logistic Regression ───────────────────────────────────────────────────────
lr = LogisticRegression(
    max_iter=1000, random_state=42, class_weight='balanced')
lr.fit(X_train_sc, y_train)

print('Both models trained successfully.')

## 6. Model Evaluation

In [ ]:
X_sc_full = scaler.transform(X)

results = {}
for name, model in [('Random Forest', rf), ('Logistic Regression', lr)]:
    yp    = model.predict(X_test_sc)
    yprob = model.predict_proba(X_test_sc)[:, 1]
    cv    = cross_val_score(model, X_sc_full, y, cv=5, scoring='accuracy').mean()
    results[name] = dict(
        accuracy=accuracy_score(y_test, yp),
        auc=roc_auc_score(y_test, yprob),
        cv=cv, yp=yp, yprob=yprob
    )
    print(f'\n{'='*50}')
    print(f'{name}')
    print(f'  Accuracy : {results[name]["accuracy"]:.4f}')
    print(f'  ROC-AUC  : {results[name]["auc"]:.4f}')
    print(f'  CV Acc   : {cv:.4f}')
    print(classification_report(y_test, yp, target_names=['No Disease','Disease']))

In [ ]:
# ROC Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colours = {'Random Forest':'#e74c3c', 'Logistic Regression':'#3498db'}

for ax, (name, res) in zip(axes, results.items()):
    fpr, tpr, _ = roc_curve(y_test, res['yprob'])
    ax.plot(fpr, tpr, color=colours[name], lw=2,
            label=f'AUC = {res["auc"]:.3f}')
    ax.plot([0,1],[0,1],'k--',lw=1.5)
    ax.fill_between(fpr, tpr, alpha=0.10, color=colours[name])
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC Curve — {name}')
    ax.legend(loc='lower right')

plt.suptitle('ROC Curves', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (name, res) in zip(axes, results.items()):
    cm   = confusion_matrix(y_test, res['yp'])
    disp = ConfusionMatrixDisplay(cm, display_labels=['No Disease','Disease'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'Confusion Matrix — {name}')
plt.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Performance summary table
from sklearn.metrics import precision_score, recall_score, f1_score
summary = []
for name, res in results.items():
    summary.append({
        'Model'    : name,
        'Accuracy' : f'{res["accuracy"]*100:.1f}%',
        'ROC-AUC'  : f'{res["auc"]:.4f}',
        'CV Acc'   : f'{res["cv"]*100:.1f}%',
        'Precision': f'{precision_score(y_test, res["yp"]):.3f}',
        'Recall'   : f'{recall_score(y_test, res["yp"]):.3f}',
        'F1-Score' : f'{f1_score(y_test, res["yp"]):.3f}',
    })
pd.DataFrame(summary)

## 7. Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Random Forest — Gini importance
rf_imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)
bar_c  = ['#e74c3c' if v >= rf_imp.median() else '#3498db' for v in rf_imp]
rf_imp.plot(kind='barh', color=bar_c, ax=axes[0], edgecolor='none')
axes[0].set_xlabel('Gini Importance'); axes[0].set_title('Random Forest Feature Importances')
patches = [mpatches.Patch(color='#e74c3c', label='Above median'),
           mpatches.Patch(color='#3498db', label='Below median')]
axes[0].legend(handles=patches, fontsize=8)

# Logistic Regression — |coefficient| importance
lr_imp = pd.Series(np.abs(lr.coef_[0]), index=FEATURES).sort_values(ascending=True)
bar_c2 = ['#e74c3c' if v >= lr_imp.median() else '#3498db' for v in lr_imp]
lr_imp.plot(kind='barh', color=bar_c2, ax=axes[1], edgecolor='none')
axes[1].set_xlabel('|Coefficient|'); axes[1].set_title('Logistic Regression Feature Importances')
patches2 = [mpatches.Patch(color='#e74c3c', label='Above median'),
            mpatches.Patch(color='#3498db', label='Below median')]
axes[1].legend(handles=patches2, fontsize=8)

plt.suptitle('Feature Importances — Both Models', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print('\nTop 5 RF features:')
print(rf_imp.sort_values(ascending=False).head().round(4))

## 8. Live Prediction Demo

In [ ]:
def risk_level(prob):
    return 'High' if prob >= 0.60 else ('Moderate' if prob >= 0.30 else 'Low')

patients = [
    {'label': 'High-risk male (62, asymptomatic CP)',
     'data': dict(age=62,sex=1,cp=3,trestbps=140,chol=268,fbs=0,
                  restecg=2,thalach=142,exang=1,oldpeak=3.0,slope=1,ca=2,thal=2)},
    {'label': 'Low-risk female (35, typical angina)',
     'data': dict(age=35,sex=0,cp=0,trestbps=112,chol=178,fbs=0,
                  restecg=0,thalach=185,exang=0,oldpeak=0.0,slope=0,ca=0,thal=1)},
    {'label': 'Moderate-risk male (55, non-anginal)',
     'data': dict(age=55,sex=1,cp=2,trestbps=132,chol=220,fbs=0,
                  restecg=1,thalach=155,exang=0,oldpeak=1.4,slope=1,ca=1,thal=3)},
]

print('='*65)
print(f'{"Patient Profile":<42} {"RF Prob":>8} {"LR Prob":>8} {"Level":>10}')
print('='*65)
for p in patients:
    X_p = np.array([[p['data'][f] for f in FEATURES]])
    X_p_sc = scaler.transform(X_p)
    rf_p   = rf.predict_proba(X_p_sc)[0][1]
    lr_p   = lr.predict_proba(X_p_sc)[0][1]
    print(f'{p["label"]:<42} {rf_p:>7.1%} {lr_p:>8.1%} {risk_level(rf_p):>10}')
print('='*65)

## 9. Key Findings & Actionable Recommendations

### 📌 Key Findings

| # | Finding | Detail |
|---|---------|--------|
| 1 | **Best Model** | Random Forest — ≥86% accuracy, ≥0.89 AUC |
| 2 | **Top Features** | `ca` (fluoroscopy vessels) · `oldpeak` (ST depression) · `thalach` (max HR) |
| 3 | **Highest Risk Group** | Males aged 55–65 with asymptomatic chest pain (cp=3) |
| 4 | **Paradox** | Asymptomatic patients have the *highest* disease rate (~72%) |
| 5 | **Cholesterol** | Weak standalone predictor — exercise variables are far stronger signals |
| 6 | **Gender Gap** | Males ~57% prevalence vs Females ~26% — 31-point disparity |

### 🏥 Actionable Recommendations

**Immediate (0–3 months)**
- Revise triage protocols: escalate *all* asymptomatic chest-pain patients to stress testing.
- Deploy the Streamlit app (`Parnil_Kashyap_HeartAttackRiskAnalysis.py`) to cardiology workstations.

**Short-term (3–6 months)**
- Introduce age-40 baseline cardiac workup (ECG + ST-depression measurement).
- Apply gender-differentiated risk alert thresholds for males 50–65.

**Long-term (6–18 months)**
- Mandate complete `ca` + `thal` data entry in all referral forms.
- Integrate the model into the EMR for automated point-of-care risk flagging.
- Add SHAP waterfall charts for per-patient explainability.
- Validate on Hungarian, Swiss, Virginia datasets before clinical deployment.

### 🏗️ Architecture
```
data/data.csv
     ↓
Parnil_Kashyap_HeartAttackRiskAnalysis.py
  ├── load_data()      — preprocessing pipeline
  ├── get_models()     — trains RF + LR; saves/loads model.pkl, scaler.pkl
  ├── predict_risk()   — inline inference (no API needed)
  ├── Tab 1            — Live Patient Risk Predictor
  ├── Tab 2            — Decision Dashboard (EDA + evaluation)
  └── Tab 3            — Summary & Recommendations
```

In [ ]:
# Persist artefacts (mirrors master app behaviour)
os.makedirs('model', exist_ok=True)
joblib.dump(rf,      'model/model.pkl')
joblib.dump(scaler,  'model/scaler.pkl')
joblib.dump(FEATURES,'model/features.pkl')
print('Saved: model/model.pkl  |  model/scaler.pkl  |  model/features.pkl')